In [1]:
import re

def valid_email(email):

    pattern = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"

    return bool(re.fullmatch(pattern, email))


def process_registrations():

    department_data = {

        "CSE": """1965260181,Kaviya,CSE,kaviya@example.com,Debugging;Web Design
1965260182,Arun,CSE,arun.example.com,Web Design
1965260183,Meena,CSE,meena@example.com,Robotics
""",

        "IT": """1965260181,Kaviya,IT,kaviya@example.com,Robotics
1965260184,Ravi,IT,ravi@example.com,Debugging
damaged,line
""",

        "ECE": """1965260185,Divya,ECE,divya@example.com,Robotics;Web Design
""",

        "EEE": """1965260186,John,EEE,john@example.com,Debugging
"""
    }

    seen = set()
    clean = []
    event_lists = {}
    errors = []
    duplicate_count = 0

    for department in ("CSE", "IT", "ECE", "EEE", "MECH"):

        if department not in department_data:
            errors.append(f"{department}.txt: missing department file")
            continue

        data = department_data[department]

        for line_no, raw in enumerate(data.splitlines(), 1):

            if not raw.strip():
                continue

            fields = [x.strip() for x in raw.split(",")]

            if len(fields) != 5:
                errors.append(
                    f"{department}.txt:{line_no}: damaged line"
                )
                continue

            reg, name, dept, email, events_text = fields

            if not reg or not name or dept != department:
                errors.append(
                    f"{department}.txt:{line_no}: invalid department or field"
                )
                continue

            if not valid_email(email):
                errors.append(
                    f"{department}.txt:{line_no}: invalid email {email!r}"
                )
                continue

            if reg in seen:
                duplicate_count += 1
                errors.append(
                    f"{department}.txt:{line_no}: duplicate registration {reg}"
                )
                continue

            seen.add(reg)

            events = [
                event.strip()
                for event in events_text.split(";")
                if event.strip()
            ]

            clean.append((reg, name, dept, email, events))

            for event in events:
                event_lists.setdefault(event, []).append(
                    (reg, name, dept)
                )

    print("ACCEPTED REGISTRATIONS:", len(clean))
    print("DUPLICATES REJECTED:", duplicate_count)

    print("\nCLEAN MASTER RECORDS")

    for record in clean:
        print(record)

    print("\nEVENT-WISE PARTICIPANTS")

    for event, people in event_lists.items():
        print(event, ":", people)

    print("\nSUMMARY")

    print("Total Events:", len(event_lists))

    for event, people in sorted(event_lists.items()):
        print(f"{event}: {len(people)} participant(s)")

    print("\nERROR LOG")

    for error in errors:
        print(error)


if __name__ == "__main__":
    process_registrations()

ACCEPTED REGISTRATIONS: 5
DUPLICATES REJECTED: 1

CLEAN MASTER RECORDS
('1965260181', 'Kaviya', 'CSE', 'kaviya@example.com', ['Debugging', 'Web Design'])
('1965260183', 'Meena', 'CSE', 'meena@example.com', ['Robotics'])
('1965260184', 'Ravi', 'IT', 'ravi@example.com', ['Debugging'])
('1965260185', 'Divya', 'ECE', 'divya@example.com', ['Robotics', 'Web Design'])
('1965260186', 'John', 'EEE', 'john@example.com', ['Debugging'])

EVENT-WISE PARTICIPANTS
Debugging : [('1965260181', 'Kaviya', 'CSE'), ('1965260184', 'Ravi', 'IT'), ('1965260186', 'John', 'EEE')]
Web Design : [('1965260181', 'Kaviya', 'CSE'), ('1965260185', 'Divya', 'ECE')]
Robotics : [('1965260183', 'Meena', 'CSE'), ('1965260185', 'Divya', 'ECE')]

SUMMARY
Total Events: 3
Debugging: 3 participant(s)
Robotics: 2 participant(s)
Web Design: 2 participant(s)

ERROR LOG
CSE.txt:2: invalid email 'arun.example.com'
IT.txt:1: duplicate registration 1965260181
IT.txt:3: damaged line
MECH.txt: missing department file
